# Comprehensive Failure Mode Signal Analysis

**Purpose:** Deep dive into subjects with F1 < 80% to understand why models fail

**Problem Subjects:**
- **S31**: Trans=78.79%, CNN=80.00% (Both struggle)
- **S38**: Trans=78.82%, CNN=74.84% (Both struggle, Trans slightly better)
- **S46**: Trans=88.05%, CNN=76.50% (CNN struggles, Trans OK)

**Analysis Sections:**
1. Raw Signal Visualization (Accelerometer + Gyroscope)
2. Signal Statistics Comparison
3. Kalman-filtered Orientation Analysis
4. Fall vs ADL Pattern Comparison
5. Model-specific Failure Patterns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import signal
from scipy.spatial.transform import Rotation
import warnings
warnings.filterwarnings('ignore')

# Setup paths
PROJECT_ROOT = Path('/home/abheekp/FusionTransformer')
DATA_DIR = PROJECT_ROOT / 'data'
ANALYSIS_DIR = PROJECT_ROOT / 'notebooks' / 'analysis'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 120

# Activity definitions
ACTIVITIES = {
    1: 'Drinking', 2: 'PickUp', 3: 'Jacket', 5: 'Sweeping',
    6: 'Washing', 7: 'Waving', 8: 'Walking', 9: 'SitStand',
    10: 'Backfall', 11: 'Frontfall', 12: 'Leftfall', 13: 'Rightfall', 14: 'Rotatefall'
}
FALL_ACTIVITIES = [10, 11, 12, 13, 14]
ADL_ACTIVITIES = [1, 2, 3, 5, 6, 7, 8, 9]

# Problem subjects
PROBLEM_SUBJECTS = [31, 38, 46]
GOOD_SUBJECTS = [50, 55, 58]  # F1 > 95% on both models for comparison

print(f"Data directory: {DATA_DIR}")
print(f"Young data exists: {(DATA_DIR / 'young').exists()}")

## 1. Data Loading Functions

In [ ]:
def load_sensor_data(subject_id, activity_id, trial_id=1, sensor='watch', modality='accelerometer'):
    """
    Load raw sensor data for a specific trial.
    
    Args:
        subject_id: Subject number (e.g., 31)
        activity_id: Activity number (1-9: ADL, 10-14: Falls)
        trial_id: Trial number (usually 1-5)
        sensor: 'watch', 'phone', 'meta_wrist', 'meta_hip'
        modality: 'accelerometer' or 'gyroscope'
    
    Returns:
        DataFrame with columns [timestamp, x, y, z]
    """
    # Young subjects have falls (10-14), old subjects don't
    age_group = 'young' if subject_id >= 29 else 'old'
    
    filename = f"S{subject_id:02d}A{activity_id:02d}T{trial_id:02d}.csv"
    filepath = DATA_DIR / age_group / modality / sensor / filename
    
    if not filepath.exists():
        return None
    
    df = pd.read_csv(filepath, header=None, names=['timestamp', 'x', 'y', 'z'])
    
    # Convert to numeric
    for col in ['x', 'y', 'z']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['x', 'y', 'z'])
    
    # Parse timestamp
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['time_sec'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
    
    return df

def load_trial_pair(subject_id, activity_id, trial_id=1, sensor='watch'):
    """
    Load both accelerometer and gyroscope data for a trial.
    
    Returns:
        acc_df, gyro_df
    """
    acc = load_sensor_data(subject_id, activity_id, trial_id, sensor, 'accelerometer')
    gyro = load_sensor_data(subject_id, activity_id, trial_id, sensor, 'gyroscope')
    return acc, gyro

# Test loading
test_acc, test_gyro = load_trial_pair(31, 10, 1, 'watch')
if test_acc is not None:
    print(f"Loaded S31 A10 T01 (Backfall): {len(test_acc)} acc samples, {len(test_gyro) if test_gyro is not None else 0} gyro samples")
    print(f"Duration: {test_acc['time_sec'].max():.2f}s")

In [ ]:
def compute_signal_features(acc_df, gyro_df=None):
    """
    Compute signal-level features for analysis.
    
    Returns:
        dict with SMV, gyro magnitude, jerk, etc.
    """
    features = {}
    
    # Signal Magnitude Vector (SMV)
    smv = np.sqrt(acc_df['x']**2 + acc_df['y']**2 + acc_df['z']**2)
    features['smv'] = smv
    features['smv_max'] = smv.max()
    features['smv_mean'] = smv.mean()
    features['smv_std'] = smv.std()
    
    # Jerk (rate of change of acceleration)
    dt = np.diff(acc_df['time_sec'].values)
    dt[dt == 0] = 0.001  # Avoid division by zero
    jerk_x = np.diff(acc_df['x'].values) / dt
    jerk_y = np.diff(acc_df['y'].values) / dt
    jerk_z = np.diff(acc_df['z'].values) / dt
    jerk_mag = np.sqrt(jerk_x**2 + jerk_y**2 + jerk_z**2)
    features['jerk_max'] = jerk_mag.max()
    features['jerk_mean'] = jerk_mag.mean()
    
    if gyro_df is not None and len(gyro_df) > 0:
        # Gyroscope magnitude (rad/s)
        gyro_mag = np.sqrt(gyro_df['x']**2 + gyro_df['y']**2 + gyro_df['z']**2)
        features['gyro_mag'] = gyro_mag
        features['gyro_max'] = gyro_mag.max()
        features['gyro_mean'] = gyro_mag.mean()
        features['gyro_std'] = gyro_mag.std()
        
        # Convert rad/s to deg/s for interpretability
        features['gyro_max_deg'] = np.rad2deg(gyro_mag.max())
        
        # Total rotation (integral of angular velocity)
        if len(gyro_df) > 1:
            dt_gyro = np.diff(gyro_df['time_sec'].values)
            dt_gyro[dt_gyro == 0] = 0.001
            total_rot = np.sum(gyro_mag.values[:-1] * dt_gyro)
            features['total_rotation_rad'] = total_rot
            features['total_rotation_deg'] = np.rad2deg(total_rot)
    
    return features

# Test
if test_acc is not None:
    feat = compute_signal_features(test_acc, test_gyro)
    print(f"SMV max: {feat['smv_max']:.2f} m/s²")
    print(f"Gyro max: {feat.get('gyro_max_deg', 0):.2f} °/s")
    print(f"Total rotation: {feat.get('total_rotation_deg', 0):.2f}°")

## 2. Simple Orientation Estimation (Complementary Filter)

In [ ]:
def estimate_orientation_complementary(acc_df, gyro_df, alpha=0.98):
    """
    Estimate orientation using complementary filter.
    
    Combines:
    - Gyroscope integration (fast, drifts)
    - Accelerometer-derived angles (noisy, no drift)
    
    Returns:
        DataFrame with roll, pitch columns (in degrees)
    """
    # Ensure same length
    min_len = min(len(acc_df), len(gyro_df))
    acc = acc_df.iloc[:min_len].copy()
    gyro = gyro_df.iloc[:min_len].copy()
    
    # Time step
    dt = np.diff(acc['time_sec'].values, prepend=acc['time_sec'].values[0])
    dt[0] = dt[1] if len(dt) > 1 else 0.02
    dt[dt == 0] = 0.02
    
    # Accelerometer-based angles
    acc_roll = np.arctan2(acc['y'], acc['z']) * 180 / np.pi
    acc_pitch = np.arctan2(-acc['x'], np.sqrt(acc['y']**2 + acc['z']**2)) * 180 / np.pi
    
    # Initialize
    roll = np.zeros(min_len)
    pitch = np.zeros(min_len)
    roll[0] = acc_roll.iloc[0]
    pitch[0] = acc_pitch.iloc[0]
    
    # Complementary filter
    for i in range(1, min_len):
        # Gyro integration (rad/s -> deg)
        gyro_roll = roll[i-1] + gyro['x'].iloc[i] * dt[i] * 180 / np.pi
        gyro_pitch = pitch[i-1] + gyro['y'].iloc[i] * dt[i] * 180 / np.pi
        
        # Complementary filter
        roll[i] = alpha * gyro_roll + (1 - alpha) * acc_roll.iloc[i]
        pitch[i] = alpha * gyro_pitch + (1 - alpha) * acc_pitch.iloc[i]
    
    result = pd.DataFrame({
        'time_sec': acc['time_sec'].values,
        'roll': roll,
        'pitch': pitch,
        'acc_roll': acc_roll.values,
        'acc_pitch': acc_pitch.values
    })
    
    return result

# Test
if test_acc is not None and test_gyro is not None:
    ori = estimate_orientation_complementary(test_acc, test_gyro)
    print(f"Roll range: [{ori['roll'].min():.1f}°, {ori['roll'].max():.1f}°]")
    print(f"Pitch range: [{ori['pitch'].min():.1f}°, {ori['pitch'].max():.1f}°]")

## 3. Visualize Problem Subject Fall Trials

In [ ]:
def plot_trial_comprehensive(subject_id, activity_id, trial_id=1, sensor='watch'):
    """
    Create comprehensive visualization of a single trial.
    
    Shows:
    - Raw accelerometer (x, y, z)
    - Raw gyroscope (x, y, z)
    - SMV and Gyro magnitude
    - Estimated orientation (roll, pitch)
    """
    acc, gyro = load_trial_pair(subject_id, activity_id, trial_id, sensor)
    
    if acc is None:
        print(f"No data for S{subject_id} A{activity_id} T{trial_id}")
        return None
    
    activity_name = ACTIVITIES.get(activity_id, f'A{activity_id}')
    is_fall = activity_id in FALL_ACTIVITIES
    
    fig, axes = plt.subplots(4, 2, figsize=(18, 14))
    fig.suptitle(f'Subject {subject_id} - {activity_name} (Trial {trial_id})\n{"FALL" if is_fall else "ADL"}', 
                 fontsize=14, fontweight='bold')
    
    # Colors
    colors = {'x': '#E74C3C', 'y': '#27AE60', 'z': '#3498DB'}
    
    # 1. Raw Accelerometer
    ax = axes[0, 0]
    ax.plot(acc['time_sec'], acc['x'], color=colors['x'], label='X', alpha=0.8)
    ax.plot(acc['time_sec'], acc['y'], color=colors['y'], label='Y', alpha=0.8)
    ax.plot(acc['time_sec'], acc['z'], color=colors['z'], label='Z', alpha=0.8)
    ax.set_ylabel('Acceleration (m/s²)')
    ax.set_title('Raw Accelerometer')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # 2. Raw Gyroscope
    ax = axes[0, 1]
    if gyro is not None:
        ax.plot(gyro['time_sec'], np.rad2deg(gyro['x']), color=colors['x'], label='X', alpha=0.8)
        ax.plot(gyro['time_sec'], np.rad2deg(gyro['y']), color=colors['y'], label='Y', alpha=0.8)
        ax.plot(gyro['time_sec'], np.rad2deg(gyro['z']), color=colors['z'], label='Z', alpha=0.8)
        ax.set_ylabel('Angular Velocity (°/s)')
        ax.set_title('Raw Gyroscope')
        ax.legend(loc='upper right')
    else:
        ax.text(0.5, 0.5, 'No gyroscope data', ha='center', va='center')
    ax.grid(True, alpha=0.3)
    
    # 3. SMV (Signal Magnitude Vector)
    ax = axes[1, 0]
    smv = np.sqrt(acc['x']**2 + acc['y']**2 + acc['z']**2)
    ax.plot(acc['time_sec'], smv, color='purple', linewidth=1.5)
    ax.axhline(y=9.81, color='gray', linestyle='--', alpha=0.5, label='1g')
    ax.axhline(y=20, color='red', linestyle='--', alpha=0.5, label='Impact threshold')
    ax.fill_between(acc['time_sec'], 0, smv, alpha=0.2, color='purple')
    ax.set_ylabel('SMV (m/s²)')
    ax.set_title(f'Signal Magnitude Vector (max={smv.max():.1f} m/s²)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # 4. Gyroscope Magnitude
    ax = axes[1, 1]
    if gyro is not None:
        gyro_mag = np.sqrt(gyro['x']**2 + gyro['y']**2 + gyro['z']**2)
        gyro_mag_deg = np.rad2deg(gyro_mag)
        ax.plot(gyro['time_sec'], gyro_mag_deg, color='orange', linewidth=1.5)
        ax.fill_between(gyro['time_sec'], 0, gyro_mag_deg, alpha=0.2, color='orange')
        ax.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='100°/s threshold')
        ax.set_ylabel('Angular Velocity Magnitude (°/s)')
        ax.set_title(f'Gyro Magnitude (max={gyro_mag_deg.max():.1f}°/s)')
        ax.legend(loc='upper right')
    else:
        ax.text(0.5, 0.5, 'No gyroscope data', ha='center', va='center')
    ax.grid(True, alpha=0.3)
    
    # 5. Estimated Orientation (Roll)
    ax = axes[2, 0]
    if gyro is not None:
        ori = estimate_orientation_complementary(acc, gyro)
        ax.plot(ori['time_sec'], ori['roll'], color='blue', linewidth=1.5, label='Fused Roll')
        ax.plot(ori['time_sec'], ori['acc_roll'], color='gray', alpha=0.5, linewidth=1, label='Acc-only Roll')
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        ax.axhline(y=90, color='red', linestyle='--', alpha=0.3)
        ax.axhline(y=-90, color='red', linestyle='--', alpha=0.3)
        ax.set_ylabel('Roll (°)')
        ax.set_title(f'Roll Angle (range: {ori["roll"].max()-ori["roll"].min():.1f}°)')
        ax.legend(loc='upper right')
    else:
        ax.text(0.5, 0.5, 'No orientation data', ha='center', va='center')
    ax.grid(True, alpha=0.3)
    
    # 6. Estimated Orientation (Pitch)
    ax = axes[2, 1]
    if gyro is not None:
        ax.plot(ori['time_sec'], ori['pitch'], color='green', linewidth=1.5, label='Fused Pitch')
        ax.plot(ori['time_sec'], ori['acc_pitch'], color='gray', alpha=0.5, linewidth=1, label='Acc-only Pitch')
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        ax.axhline(y=90, color='red', linestyle='--', alpha=0.3)
        ax.axhline(y=-90, color='red', linestyle='--', alpha=0.3)
        ax.set_ylabel('Pitch (°)')
        ax.set_title(f'Pitch Angle (range: {ori["pitch"].max()-ori["pitch"].min():.1f}°)')
        ax.legend(loc='upper right')
    else:
        ax.text(0.5, 0.5, 'No orientation data', ha='center', va='center')
    ax.grid(True, alpha=0.3)
    
    # 7. Jerk (acceleration derivative)
    ax = axes[3, 0]
    dt = np.diff(acc['time_sec'].values)
    dt[dt == 0] = 0.001
    jerk = np.sqrt(np.diff(acc['x'])**2 + np.diff(acc['y'])**2 + np.diff(acc['z'])**2) / dt
    ax.plot(acc['time_sec'].values[1:], jerk, color='darkred', linewidth=1)
    ax.fill_between(acc['time_sec'].values[1:], 0, jerk, alpha=0.2, color='darkred')
    ax.set_ylabel('Jerk (m/s³)')
    ax.set_xlabel('Time (s)')
    ax.set_title(f'Jerk Magnitude (max={jerk.max():.1f} m/s³)')
    ax.grid(True, alpha=0.3)
    
    # 8. Statistics summary
    ax = axes[3, 1]
    ax.axis('off')
    
    feat = compute_signal_features(acc, gyro)
    stats_text = f"""
    Signal Statistics
    ─────────────────────────────
    Duration: {acc['time_sec'].max():.2f} s
    Samples: {len(acc)} acc, {len(gyro) if gyro is not None else 0} gyro
    
    Accelerometer:
      SMV max: {feat['smv_max']:.2f} m/s²
      SMV mean: {feat['smv_mean']:.2f} m/s²
      Jerk max: {feat['jerk_max']:.1f} m/s³
    
    Gyroscope:
      Max angular vel: {feat.get('gyro_max_deg', 0):.1f}°/s
      Total rotation: {feat.get('total_rotation_deg', 0):.1f}°
    
    Orientation:
      Roll range: {ori['roll'].max()-ori['roll'].min():.1f}° (if avail)
      Pitch range: {ori['pitch'].max()-ori['pitch'].min():.1f}° (if avail)
    """
    ax.text(0.1, 0.95, stats_text, transform=ax.transAxes, fontsize=11,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    return fig, feat

# Test with S31 backfall
fig, feat = plot_trial_comprehensive(31, 10, 1, 'watch')
plt.show()

## 4. Analyze All Fall Trials for Problem Subjects

In [ ]:
def analyze_subject_falls(subject_id):
    """
    Analyze all fall trials for a subject.
    
    Returns:
        DataFrame with features for each trial
    """
    results = []
    
    for activity_id in FALL_ACTIVITIES:
        for trial_id in range(1, 6):  # Up to 5 trials
            acc, gyro = load_trial_pair(subject_id, activity_id, trial_id, 'watch')
            
            if acc is not None:
                feat = compute_signal_features(acc, gyro)
                feat['subject'] = subject_id
                feat['activity'] = activity_id
                feat['activity_name'] = ACTIVITIES.get(activity_id, f'A{activity_id}')
                feat['trial'] = trial_id
                feat['duration'] = acc['time_sec'].max()
                feat['n_samples'] = len(acc)
                results.append(feat)
    
    return pd.DataFrame(results)

# Analyze problem subjects
problem_analysis = {}
for subj in PROBLEM_SUBJECTS:
    df = analyze_subject_falls(subj)
    problem_analysis[subj] = df
    print(f"\nS{subj}: {len(df)} fall trials")
    if len(df) > 0:
        print(f"  SMV max range: [{df['smv_max'].min():.1f}, {df['smv_max'].max():.1f}] m/s²")
        print(f"  Gyro max range: [{df['gyro_max_deg'].min():.1f}, {df['gyro_max_deg'].max():.1f}]°/s")
        print(f"  Total rot range: [{df['total_rotation_deg'].min():.1f}, {df['total_rotation_deg'].max():.1f}]°")

In [ ]:
# Compare with good subjects
good_analysis = {}
for subj in GOOD_SUBJECTS:
    df = analyze_subject_falls(subj)
    good_analysis[subj] = df
    print(f"\nS{subj} (Good): {len(df)} fall trials")
    if len(df) > 0:
        print(f"  SMV max range: [{df['smv_max'].min():.1f}, {df['smv_max'].max():.1f}] m/s²")
        print(f"  Gyro max range: [{df['gyro_max_deg'].min():.1f}, {df['gyro_max_deg'].max():.1f}]°/s")

## 5. Compare Problem vs Good Subjects

In [ ]:
# Combine all analyses
problem_df = pd.concat([df.assign(category='Problem') for df in problem_analysis.values()], ignore_index=True)
good_df = pd.concat([df.assign(category='Good') for df in good_analysis.values()], ignore_index=True)
all_df = pd.concat([problem_df, good_df], ignore_index=True)

print(f"Total trials: {len(all_df)} (Problem: {len(problem_df)}, Good: {len(good_df)})")

In [ ]:
# Visualization: Compare distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

metrics = [
    ('smv_max', 'SMV Max (m/s²)', 'Impact Force'),
    ('gyro_max_deg', 'Gyro Max (°/s)', 'Rotation Speed'),
    ('total_rotation_deg', 'Total Rotation (°)', 'Rotation Amount'),
    ('jerk_max', 'Jerk Max (m/s³)', 'Acceleration Change'),
    ('duration', 'Duration (s)', 'Trial Length'),
    ('smv_std', 'SMV Std', 'Signal Variability')
]

for idx, (metric, xlabel, title) in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    
    if metric in all_df.columns:
        # Box plot
        problem_vals = all_df[all_df['category'] == 'Problem'][metric].dropna()
        good_vals = all_df[all_df['category'] == 'Good'][metric].dropna()
        
        bp = ax.boxplot([problem_vals, good_vals], labels=['Problem\n(S31,S38,S46)', 'Good\n(S50,S55,S58)'],
                       patch_artist=True)
        bp['boxes'][0].set_facecolor('#E74C3C')
        bp['boxes'][1].set_facecolor('#27AE60')
        
        ax.set_ylabel(xlabel)
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        
        # Add means
        ax.scatter([1], [problem_vals.mean()], color='darkred', s=100, marker='D', zorder=5, label=f'Mean: {problem_vals.mean():.1f}')
        ax.scatter([2], [good_vals.mean()], color='darkgreen', s=100, marker='D', zorder=5, label=f'Mean: {good_vals.mean():.1f}')

plt.suptitle('Signal Characteristics: Problem vs Good Subjects (Fall Trials Only)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'problem_vs_good_signal_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Detailed Per-Subject Fall Trial Visualizations

In [ ]:
# Visualize all fall types for each problem subject
print("="*70)
print("GENERATING DETAILED VISUALIZATIONS FOR PROBLEM SUBJECTS")
print("="*70)

for subj in PROBLEM_SUBJECTS:
    print(f"\n{'='*70}")
    print(f"SUBJECT {subj}")
    print(f"{'='*70}")
    
    for activity_id in FALL_ACTIVITIES:
        activity_name = ACTIVITIES.get(activity_id, f'A{activity_id}')
        
        # Try to load first trial
        acc, gyro = load_trial_pair(subj, activity_id, 1, 'watch')
        
        if acc is not None:
            print(f"\n--- {activity_name} ---")
            fig, feat = plot_trial_comprehensive(subj, activity_id, 1, 'watch')
            
            # Save figure
            fig.savefig(FIGURES_DIR / f'S{subj}_{activity_name.lower()}_signal_analysis.png', 
                       dpi=200, bbox_inches='tight')
            plt.show()
            plt.close(fig)

## 7. Model Results Heatmap

In [ ]:
# Load model scores
RESULTS_DIR = ANALYSIS_DIR / 'results_dec22'
trans_df = pd.read_csv(RESULTS_DIR / 'transformer' / 'scores.csv')
cnn_df = pd.read_csv(RESULTS_DIR / 'cnn' / 'scores.csv')

# Remove Average
trans_df = trans_df[trans_df['test_subject'] != 'Average'].copy()
cnn_df = cnn_df[cnn_df['test_subject'] != 'Average'].copy()

trans_df['test_subject'] = pd.to_numeric(trans_df['test_subject'])
cnn_df['test_subject'] = pd.to_numeric(cnn_df['test_subject'])

# Create comparison dataframe
comparison = pd.DataFrame({
    'Subject': trans_df['test_subject'],
    'Transformer F1': pd.to_numeric(trans_df['test_f1_score']),
    'CNN F1': pd.to_numeric(cnn_df['test_f1_score']),
    'Trans Precision': pd.to_numeric(trans_df['test_precision']),
    'Trans Recall': pd.to_numeric(trans_df['test_recall']),
    'CNN Precision': pd.to_numeric(cnn_df['test_precision']),
    'CNN Recall': pd.to_numeric(cnn_df['test_recall'])
})

comparison['Delta F1'] = comparison['Transformer F1'] - comparison['CNN F1']
comparison = comparison.sort_values('Transformer F1')

# Highlight problem subjects
comparison['Category'] = comparison['Subject'].apply(
    lambda x: 'Problem' if x in PROBLEM_SUBJECTS else ('Good' if x in GOOD_SUBJECTS else 'Normal')
)

display(comparison[comparison['Category'] == 'Problem'])

In [ ]:
# Create detailed heatmap
fig, ax = plt.subplots(figsize=(14, 10))

# Prepare heatmap data
heatmap_data = comparison[['Subject', 'Transformer F1', 'CNN F1', 'Trans Precision', 'Trans Recall', 'CNN Precision', 'CNN Recall']].copy()
heatmap_data = heatmap_data.set_index('Subject')

# Create annotations showing values
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn', 
            center=85, vmin=60, vmax=100, linewidths=0.5,
            cbar_kws={'label': 'Score (%)'}, ax=ax)

# Highlight problem subjects
for i, subj in enumerate(heatmap_data.index):
    if subj in PROBLEM_SUBJECTS:
        ax.add_patch(plt.Rectangle((0, i), len(heatmap_data.columns), 1, 
                                    fill=False, edgecolor='red', linewidth=3))

ax.set_title('Per-Subject Model Performance Heatmap\n(Red boxes = Problem subjects F1 < 80%)', fontsize=14)
ax.set_ylabel('Subject ID')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'per_subject_performance_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Summary & Key Insights

In [ ]:
print("="*70)
print("FAILURE MODE ANALYSIS SUMMARY")
print("="*70)

print("""
PROBLEM SUBJECTS (F1 < 80%):
───────────────────────────────────────────────────────────────────────
""")

for subj in PROBLEM_SUBJECTS:
    t_f1 = comparison[comparison['Subject'] == subj]['Transformer F1'].values[0]
    c_f1 = comparison[comparison['Subject'] == subj]['CNN F1'].values[0]
    t_prec = comparison[comparison['Subject'] == subj]['Trans Precision'].values[0]
    t_rec = comparison[comparison['Subject'] == subj]['Trans Recall'].values[0]
    c_prec = comparison[comparison['Subject'] == subj]['CNN Precision'].values[0]
    c_rec = comparison[comparison['Subject'] == subj]['CNN Recall'].values[0]
    
    df = problem_analysis.get(subj, pd.DataFrame())
    
    print(f"\nS{subj}:")
    print(f"  Transformer: {t_f1:.1f}% F1 (Prec={t_prec:.1f}%, Rec={t_rec:.1f}%)")
    print(f"  CNN:         {c_f1:.1f}% F1 (Prec={c_prec:.1f}%, Rec={c_rec:.1f}%)")
    
    if len(df) > 0:
        print(f"  Signal Characteristics:")
        print(f"    - SMV max: {df['smv_max'].mean():.1f} ± {df['smv_max'].std():.1f} m/s²")
        print(f"    - Gyro max: {df['gyro_max_deg'].mean():.1f} ± {df['gyro_max_deg'].std():.1f}°/s")
        print(f"    - Total rotation: {df['total_rotation_deg'].mean():.1f}°")

print("""
───────────────────────────────────────────────────────────────────────
KEY OBSERVATIONS:

1. S31: High recall (100%) but low precision (65-67%) on BOTH models
   -> Too many false positives (ADLs classified as falls)
   
2. S38: Lower recall (66-77%) - both models miss some falls
   -> Likely has "gentle" falls with low SMV/rotation
   
3. S46: CNN struggles (76.5%) but Transformer OK (88%)
   -> Transformer's attention mechanism handles this subject better

HYPOTHESES:
- Problem subjects may have less distinct fall signatures
- Lower impact force (SMV) and rotation speed
- More ambiguous ADL patterns that look like falls
""")

In [ ]:
print("\nFigures saved to:", FIGURES_DIR)
print("\nGenerated files:")
for f in sorted(FIGURES_DIR.glob('*.png')):
    print(f"  - {f.name}")